# DuckDB no Notisbook

Este exemplo usa o kernel DuckDB e o catálogo `notislake` para criar uma tabela Iceberg, gravar um Parquet no lake e lê-lo de volta.

In [ ]:
CREATE SCHEMA IF NOT EXISTS notislake.bronze

Criamos a tabela Iceberg `notislake.bronze.duckdb_employees` com oito linhas geradas por `range()`. O `DROP TABLE IF EXISTS` permite reexecutar a célula.

In [ ]:
DROP TABLE IF EXISTS notislake.bronze.duckdb_employees;

CREATE TABLE notislake.bronze.duckdb_employees AS
SELECT id::BIGINT AS id, nome, area
FROM (VALUES
    (1, 'Romerito', 'Platform Engineering'),
    (2, 'Theodoro', 'Data Engineering'),
    (3, 'Miguel', 'AI Engineering'),
    (4, 'Nina', 'Coffee Engineering'),
    (5, 'Niko', 'Security Engineering'),
    (6, 'Leonor', 'Governance'),
    (7, 'Ana', 'Data Engineering'),
    (8, 'Bruno', 'Platform Engineering')
) AS t(id, nome, area)

Exportamos a tabela para um arquivo Parquet na zona `landing` do notislake, acessada por path `s3a://`.

In [ ]:
COPY (SELECT * FROM notislake.bronze.duckdb_employees) TO 's3a://notislake/landing/duckdb_employees.parquet' (FORMAT parquet)

Lemos o Parquet de volta direto do path, sem passar pelo catálogo.

In [ ]:
SELECT * FROM read_parquet('s3a://notislake/landing/duckdb_employees.parquet')

Por fim, consultamos a tabela Iceberg pelo catálogo.

In [ ]:
SELECT * FROM notislake.bronze.duckdb_employees